# cAST-Scope — test rapide, T4, 50 tâches

Version allégée de `colab_benchmark.ipynb` pour économiser les unités de calcul Colab : GPU **T4** (pas A100), **50 tâches** RepoEval seulement (pas CrossCodeEval), **un seul modèle** (StarCoder2-7B). Objectif : confirmer que tout tourne jusqu'au tableau final avant d'investir plus d'unités dans un run complet.

**Important** : sélectionnez `Runtime > Change runtime type > T4 GPU` avant de lancer la cellule 0 (le T4 consomme moins d'unités/heure que l'A100, largement suffisant pour 50 tâches).

Utilise `%run` (pas `!python`) — un sous-processus `!python` se bloquait sur Colab pendant le chunking AST (raison non identifiée), `%run` exécute dans le kernel et contourne ça.

## 0. Vérifier le GPU (doit afficher T4)

In [ ]:
!nvidia-smi

## 1. Cloner (ou mettre à jour) le dépôt + installer les dépendances

In [ ]:
import os

if os.path.isdir('/content/cAST-state'):
    %cd /content/cAST-state
    !git pull
else:
    !git clone --depth 1 https://github.com/Robertkiza0/cAST-state.git /content/cAST-state
    %cd /content/cAST-state

# Installer transformers/accelerate dans un kernel deja demarre peut le faire
# planter (conflit numpy/dependances bas niveau deja chargees en memoire) --
# probleme connu de Colab (voir googlecolab/colabtools#1753 et similaires).
# Le correctif standard : redemarrer le kernel juste apres l'installation,
# AVANT d'executer quoi que ce soit d'autre. On le fait ici automatiquement
# (une seule fois par VM, via un fichier sentinelle) plutot que de compter
# sur un clic manuel qui a ete a l'origine de plusieurs sessions de debug.
SENTINEL = '/content/.cast_state_installed'

if not os.path.exists(SENTINEL):
    !pip install -q -r requirements.txt
    !pip install -q transformers accelerate editdistance
    open(SENTINEL, 'w').close()
    print()
    print("=== Paquets installes. Redemarrage AUTOMATIQUE du kernel maintenant (normal, pas une erreur) ===")
    print("=== Une fois redemarre : RELANCEZ CETTE MEME CELLULE UNE FOIS (rien a reinstaller, ca passera direct a la suite), puis continuez normalement. ===")
    import time
    time.sleep(2)
    os._exit(0)

print()
!echo "=== Commit actif : $(git rev-parse --short HEAD) — $(git log -1 --format=%s) ==="


## 1bis. (Optionnel) Token Hugging Face

In [ ]:
from google.colab import userdata
import os

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN chargé.")
except Exception:
    print("Pas de secret HF_TOKEN configuré — pas grave, pas nécessaire pour un modèle public.")


## 2. Télécharger les données RepoEval (tâches + 8 dépôts réels)

Idempotent : si déjà présent, ne retélécharge rien.

In [ ]:
import os
import zipfile

if os.path.isdir('data/repos_source') and os.listdir('data/repos_source'):
    print('RepoEval déjà présent, rien à faire.')
else:
    !rm -rf codet_src
    !git clone --no-checkout --depth 1 https://github.com/microsoft/CodeT.git codet_src
    %cd codet_src
    !git sparse-checkout init --cone
    !git sparse-checkout set RepoCoder
    !git checkout main
    %cd ..

    with zipfile.ZipFile('codet_src/RepoCoder/datasets/datasets.zip') as z:
        z.extractall('datasets rapo')
    with zipfile.ZipFile('codet_src/RepoCoder/repositories/line_and_api_level.zip') as z:
        z.extractall('data/repos_source')

    print('RepoEval : dataset et dépôts extraits.')


## 3. Test factice (générateur stub, valide le pipeline sans consommer d'unités GPU)

In [ ]:
%run -i run_benchmark.py --dataset repoeval --n-tasks 10 --generator stub


## 4. Run réel — StarCoder2-7B, 50 tâches, T4

`--tasks-per-repo 7` pour répartir les 50 tâches sur les 8 dépôts disponibles (50/8 ≈ 7) plutôt que de les prendre toutes dans le même dépôt (comportement par défaut avec `--tasks-per-repo 10` si le premier dépôt en a assez).

In [ ]:
%run -i run_benchmark.py --dataset repoeval --n-tasks 50 --tasks-per-repo 7 --generator hf --model-name bigcode/starcoder2-7b --device cuda


## Notes

- Pass@1 == Exact Match ici (pas de harnais d'exécution — voir `metrics.py:compute_pass_at_1`).
- Si ça tourne bien jusqu'au tableau final, passez à `colab_benchmark.ipynb` (A100, 300 tâches, RepoEval+CrossCodeEval, StarCoder2+CodeLlama) pour le run complet — plus cher en unités, à réserver une fois que ce test léger confirme que tout fonctionne.
- `Runtime > Restart session` si besoin de repartir propre ; toutes les cellules d'installation sont idempotentes (sûres à relancer).